# Vanilla VAE baseline for atmospheric C 1s spectra

This notebook trains a conventional fully connected Gaussian VAE on the **same self-contained HDF5 dataset and packaged physical-case split** as `prophesy_beta_vae.ipynb`. Each noisy acquisition is the input; its noise-free zero-inelastic-collision peak spectrum is the target. The decoder has a Gaussian observation model with fixed standard deviation in normalized Anscombe units, and the KL term has its standard coefficient of one. There are no peak, area, derivative, or warm-up terms.

The saved outputs have been cleared because the previous run used a different target. Run the cells in order to train and evaluate this version. Run its cells in order after reviewing the settings. Required packages: PyTorch, NumPy, h5py, and Matplotlib. Keep the notebook in the `spectraVAE` project folder so it can import `atmospheric_dataset.py`.

In [ ]:
from pathlib import Path
import copy
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from atmospheric_dataset import h5_manifest, load_atmospheric_c1s

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}; device={DEVICE}")

## Load the new dataset

The existing cache is used only if its HDF5 manifest matches the files on disk. Only raw HDF5 data are cached; model weights and outputs are separate. The checkpoint uses a separate baseline output directory.

In [ ]:
DATA_ROOT_OVERRIDE = None
DATA_ROOT = Path(DATA_ROOT_OVERRIDE) if DATA_ROOT_OVERRIDE else Path.cwd() / "atmospheric_c1s_h5"
if not DATA_ROOT.exists():
    raise FileNotFoundError(f"HDF5 dataset not found: {DATA_ROOT}")
CACHE_DIR = Path.cwd() / "data_cache"
CACHE_FILE = CACHE_DIR / "atmospheric_c1s_401.npz"
OUTPUT_DIR = Path.cwd() / "outputs" / "atmospheric_vanilla_vae_zero_loss"
CHECKPOINT_FILE = OUTPUT_DIR / "best_vanilla_vae_zero_loss.pt"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
current_manifest = h5_manifest(DATA_ROOT)
if len(current_manifest) < 2:
    raise FileNotFoundError(f"No indexed HDF5 cases found in {DATA_ROOT}")
cache_current = False
if CACHE_FILE.exists():
    with np.load(CACHE_FILE, allow_pickle=False) as cached:
        cache_current = "h5_manifest" in cached.files and np.array_equal(cached["h5_manifest"], current_manifest)
if cache_current:
    with np.load(CACHE_FILE, allow_pickle=False) as cached:
        dataset = {key: cached[key] for key in cached.files if key != "h5_manifest"}
else:
    dataset = load_atmospheric_c1s(DATA_ROOT, target_length=401)
    np.savez_compressed(CACHE_FILE, **dataset, h5_manifest=current_manifest)
print(f"Dataset: {DATA_ROOT}")
print(f"Acquisitions: {len(dataset['noisy']):,}; physical cases: {np.unique(dataset['case_id']).size:,}")
print(f"Energy points: {len(dataset['axis'])}; splits: {dict(zip(*np.unique(dataset['split'], return_counts=True)))}")

## Case-safe split and normalization

Five noisy acquisitions of each simulated case remain in the same split. The Anscombe scale is determined from training-case total spectra so both the noisy inputs and background-free targets use the same count scale. This baseline uses every training acquisition once per epoch.

In [ ]:
case_id = dataset["case_id"]
split = dataset["split"]
train_idx = np.flatnonzero(split == "train")
val_idx = np.flatnonzero(split == "validation")
test_idx = np.flatnonzero(split == "test")
case_sets = [set(np.unique(case_id[index])) for index in (train_idx, val_idx, test_idx)]
assert all(case_sets[i].isdisjoint(case_sets[j]) for i, j in ((0, 1), (0, 2), (1, 2)))
assert sum(map(len, (train_idx, val_idx, test_idx))) == len(case_id)

def anscombe(counts):
    return 2.0 * np.sqrt(np.maximum(counts, 0.0) + 3.0 / 8.0)

TARGET_KEY = "clean"  # SESSA zero-inelastic-collision spectrum, without transport background
TRANSFORM_SCALE = 1.05 * float(anscombe(dataset["expected_total"][train_idx]).max())
def normalize_counts(counts):
    return np.clip(anscombe(counts) / TRANSFORM_SCALE, 0.0, 1.0).astype(np.float32)
def denormalize_counts(values):
    transformed = np.clip(values, 0.0, 1.0) * TRANSFORM_SCALE
    return np.maximum((transformed / 2.0) ** 2 - 3.0 / 8.0, 0.0)

inputs = normalize_counts(dataset["noisy"])
targets = normalize_counts(dataset[TARGET_KEY])
BATCH_SIZE = 512 if DEVICE.type == "cuda" else 256

def make_loader(index, shuffle=False):
    pairs = TensorDataset(torch.from_numpy(inputs[index]), torch.from_numpy(targets[index]))
    return DataLoader(pairs, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0,
                      pin_memory=DEVICE.type == "cuda")

train_loader = make_loader(train_idx, shuffle=True)
val_loader = make_loader(val_idx)
test_loader = make_loader(test_idx)
print(f"Rows: train={len(train_idx):,}, validation={len(val_idx):,}, test={len(test_idx):,}")
print(f"Cases: train={len(case_sets[0]):,}, validation={len(case_sets[1]):,}, test={len(case_sets[2]):,}")
print(f"Anscombe scale: {TRANSFORM_SCALE:.4f}")

## Vanilla Gaussian VAE

A ReLU multilayer perceptron maps 401 intensities to 12 Gaussian latent dimensions. The decoder predicts the Gaussian mean in transformed intensity units. The loss is the usual negative ELBO (up to the Gaussian constant): summed squared error divided by `2 × observation_sigma²`, plus KL divergence to the standard normal, both averaged per spectrum. `OBSERVATION_SIGMA` is a declared likelihood assumption, so record it when comparing runs. The model is selected using deterministic decoding from the encoder mean for stable validation.

In [ ]:
LATENT_DIM = 12
OBSERVATION_SIGMA = 0.03
EPOCHS = 150
LEARNING_RATE = 3e-4
PATIENCE = 20
MIN_DELTA = 0.01

class VanillaVAE(nn.Module):
    def __init__(self, input_dim=401, latent_dim=12):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
        )
        self.mu = nn.Linear(64, latent_dim)
        self.logvar = nn.Linear(64, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, input_dim),
        )

    def encode(self, x):
        features = self.encoder(x)
        return self.mu(features), self.logvar(features)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        return self.decode(z), mu, logvar

def loss_parts(reconstruction, target, mu, logvar):
    squared = (reconstruction - target).square().sum(dim=1)
    gaussian_nll = (squared / (2.0 * OBSERVATION_SIGMA ** 2)).mean()
    kl = (-0.5 * (1.0 + logvar - mu.square() - logvar.exp()).sum(dim=1)).mean()
    return gaussian_nll + kl, gaussian_nll, kl

model = VanillaVAE(input_dim=len(dataset["axis"]), latent_dim=LATENT_DIM).to(DEVICE)
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## Train and retain the best validation checkpoint

Training samples the latent posterior. Validation decodes its mean to avoid selection noise; the displayed validation score is therefore a deterministic selection score, rather than a Monte Carlo estimate of the ELBO.

In [ ]:
@torch.no_grad()
def evaluate(loader):
    model.eval()
    totals = np.zeros(3, dtype=np.float64)
    count = 0
    for x, target in loader:
        x, target = x.to(DEVICE), target.to(DEVICE)
        mu, logvar = model.encode(x)
        values = loss_parts(model.decode(mu), target, mu, logvar)
        totals += np.array([value.item() for value in values]) * len(x)
        count += len(x)
    return totals / count

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
history = {"train_total": [], "train_reconstruction": [], "train_kl": [],
           "val_total": [], "val_reconstruction": [], "val_kl": []}
best_score = float("inf")
best_state = None
stale_epochs = 0
start_time = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    totals = np.zeros(3, dtype=np.float64)
    count = 0
    for x, target in train_loader:
        x, target = x.to(DEVICE), target.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        reconstruction, mu, logvar = model(x)
        values = loss_parts(reconstruction, target, mu, logvar)
        values[0].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        totals += np.array([value.item() for value in values]) * len(x)
        count += len(x)
    train_values = totals / count
    val_values = evaluate(val_loader)
    for group, values in (("train", train_values), ("val", val_values)):
        for name, value in zip(("total", "reconstruction", "kl"), values):
            history[f"{group}_{name}"].append(value)
    if np.isfinite(val_values[0]) and val_values[0] < best_score - MIN_DELTA:
        best_score = float(val_values[0])
        best_state = copy.deepcopy(model.state_dict())
        stale_epochs = 0
        torch.save({"model_state_dict": best_state, "latent_dim": LATENT_DIM,
                    "observation_sigma": OBSERVATION_SIGMA, "transform_scale": TRANSFORM_SCALE,
                    "target_key": TARGET_KEY,
                    "h5_manifest": current_manifest.tolist(), "seed": SEED}, CHECKPOINT_FILE)
    else:
        stale_epochs += 1
    if epoch == 1 or epoch % 10 == 0:
        print(f"epoch {epoch:03d}: train={train_values[0]:.3f} "
              f"(nll={train_values[1]:.3f}, kl={train_values[2]:.3f}); "
              f"val={val_values[0]:.3f}")
    if stale_epochs >= PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break
if best_state is None:
    raise RuntimeError("Training did not produce a finite validation checkpoint")
model.load_state_dict(best_state)
print(f"Best validation score: {best_score:.3f}; time: {(time.time()-start_time)/60:.1f} min")
print(f"Checkpoint: {CHECKPOINT_FILE}")

## Baseline diagnostics on held-out physical cases

Evaluate the predicted zero-loss signal directly against the held-out zero-loss target. These scores measure denoising and background removal together. The saved β-VAE scores used the background-containing total spectrum and are not directly comparable.

The plots show the three best and three worst held-out physical cases, ranked by median normalized RMSE across each case's noisy acquisitions. One acquisition nearest that median represents each case.


In [ ]:
@torch.no_grad()
def reconstruct_mean(values, batch_size=1024):
    model.eval()
    chunks = []
    for start in range(0, len(values), batch_size):
        x = torch.from_numpy(values[start:start+batch_size]).to(DEVICE)
        mu, _ = model.encode(x)
        chunks.append(model.decode(mu).cpu().numpy())
    return np.concatenate(chunks)

prediction = denormalize_counts(reconstruct_mean(inputs[test_idx]))
target = dataset[TARGET_KEY][test_idx]
axis = dataset["axis"]
rmse = np.sqrt(np.mean((prediction-target)**2, axis=1))
nrmse = rmse / np.maximum(np.ptp(target, axis=1), 1.0)
area_error = np.abs(prediction.sum(axis=1)-target.sum(axis=1)) / np.maximum(target.sum(axis=1), 1.0)
peak_error = np.abs(prediction.max(axis=1)-target.max(axis=1)) / np.maximum(target.max(axis=1), 1.0)
position_error = np.abs(axis[np.argmax(prediction, axis=1)]-axis[np.argmax(target, axis=1)])
print(f"Deterministic test selection score: {evaluate(test_loader)[0]:.3f}")
for name, values, scale, unit in (
    ("NRMSE", nrmse, 1, ""),
    ("Area error", area_error, 100, "%"),
    ("Peak-height error", peak_error, 100, "%"),
    ("Dominant-peak position error", position_error, 1, " eV"),
):
    q = np.quantile(values, [0.5, 0.9, 0.95]) * scale
    print(f"{name:<29} p50={q[0]:.3f}{unit}  p90={q[1]:.3f}{unit}  p95={q[2]:.3f}{unit}")

test_case_ids = case_id[test_idx]
case_scores = []
case_representatives = []
for physical_case in np.unique(test_case_ids):
    positions = np.flatnonzero(test_case_ids == physical_case)
    median_error = np.median(nrmse[positions])
    case_scores.append(median_error)
    case_representatives.append(positions[np.argmin(np.abs(nrmse[positions] - median_error))])
case_scores = np.asarray(case_scores)
case_representatives = np.asarray(case_representatives)
case_order = np.argsort(case_scores)
best_cases = case_order[:3]
worst_cases = case_order[-3:][::-1]

fig, axes = plt.subplots(3, 2, figsize=(15, 11), sharex=True)
for rank, (best_case, worst_case) in enumerate(zip(best_cases, worst_cases), start=1):
    for column, case_index in enumerate((best_case, worst_case)):
        pos = case_representatives[case_index]
        row = test_idx[pos]
        ax = axes[rank - 1, column]
        ax.plot(axis, dataset["noisy"][row], color="0.7", lw=0.8, label="noisy input")
        ax.plot(axis, target[pos], color="tab:orange", lw=1.2, label="background-free target")
        ax.plot(axis, prediction[pos], color="black", lw=1.2, label="vanilla VAE output")
        label = "Best" if column == 0 else "Worst"
        ax.set_title(f"{label} #{rank} | case {case_id[row]} | case median NRMSE={case_scores[case_index]:.3f}"
                     f" | shown peak error={peak_error[pos]:.1%}")
        ax.set_ylabel("Counts")
        ax.grid(alpha=0.2)
axes[0, 0].legend(ncol=3)
for ax in axes[-1]:
    ax.set_xlabel("Binding energy (eV)")
fig.suptitle("Best and worst background-free VAE outputs on held-out cases", y=1.01)
plt.tight_layout()
plt.show()

## Decode interpolations between training cases

To inspect generation on the learned data manifold, encode one training acquisition per physical case and interpolate between nearby latent means. This section makes no independent latent draw. Decoded interpolations represent background-free peak spectra. Inspect their sharp components against the test reconstructions above.

In [ ]:
@torch.no_grad()
def encode_means(values, batch_size=1024):
    model.eval()
    chunks = []
    for start in range(0, len(values), batch_size):
        x = torch.from_numpy(values[start:start+batch_size]).to(DEVICE)
        mu, _ = model.encode(x)
        chunks.append(mu.cpu().numpy())
    return np.concatenate(chunks)

_, representative_rows = np.unique(case_id[train_idx], return_index=True)
train_representatives = train_idx[representative_rows]
latent_means = encode_means(inputs[train_representatives])
latent_std = latent_means.std(axis=0)
active = np.flatnonzero(latent_std > 0.05)
print("Std of encoder means:", np.round(latent_std, 3))
print("Active dimensions:", active.tolist())
if not len(active):
    raise RuntimeError("No active latent means; interpolation would be uninformative")
scaled = latent_means[:, active] / latent_std[active]
rng = np.random.default_rng(SEED + 2)
anchors = rng.choice(len(latent_means), size=12, replace=False)
new_latents = []
for anchor in anchors:
    candidates = rng.choice(len(latent_means), size=min(512, len(latent_means)), replace=False)
    candidates = candidates[candidates != anchor]
    distances = np.sum((scaled[candidates]-scaled[anchor])**2, axis=1)
    neighbor = rng.choice(candidates[np.argsort(distances)[:20]])
    weight = rng.uniform(0.25, 0.75)
    new_latents.append((1-weight)*latent_means[anchor] + weight*latent_means[neighbor])
with torch.no_grad():
    decoded = model.decode(torch.tensor(np.array(new_latents), dtype=torch.float32, device=DEVICE)).cpu().numpy()
generated = denormalize_counts(decoded)
fig, axes = plt.subplots(3, 4, figsize=(14, 8), sharex=True)
for ax, spectrum in zip(axes.flat, generated):
    ax.plot(axis, spectrum, lw=1.1)
    ax.grid(alpha=0.2)
for ax in axes[-1]:
    ax.set_xlabel("Binding energy (eV)")
for ax in axes[:, 0]:
    ax.set_ylabel("Counts")
fig.suptitle("Vanilla VAE decoded training-case interpolations", y=1.01)
plt.tight_layout()
plt.show()